# Deduplikasi & Stratified Split 70/15/15 — Lung Cancer MRI Dataset

Dataset asli memiliki split `train`/`validate` bawaan Kaggle dengan total 3.680 citra. Sebelum membuat split 70/15/15 sendiri, notebook ini pertama-tama **mendeteksi dan membersihkan masalah kualitas data** yang ditemukan lewat perceptual hashing (resize 64x64 + MD5 pada tiap citra):

1. **Duplikasi** — 742 dari 2.886 grup citra unik punya lebih dari satu salinan fisik. Kalau dibiarkan, salinan yang sama bisa jatuh di `train` dan `test` sekaligus (data leakage) sehingga akurasi test jadi optimis palsu.
2. **Label bertentangan** — 11 grup citra yang identik secara visual ternyata diberi label berbeda (`cancer` di satu file, `no_cancer` di file lain) oleh pembuat dataset aslinya. Ini label noise yang tidak bisa diperbaiki, jadi seluruh anggota grup tersebut (23 citra) dibuang.

Setelah dedup + drop conflicting label, tersisa **2.875 citra unik** yang baru dibagi 70/15/15 secara stratified. Temuan menarik: rasio kelas berubah dari ~51:49 (cancer:no_cancer) di data mentah menjadi **~42:58** setelah dedup — artinya kelas `cancer` di dataset asli proporsinya lebih banyak diduplikasi, membuat dataset terlihat lebih seimbang dari yang sebenarnya.


In [1]:
import hashlib
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

MANIFEST_PATH = Path("../reports/dataset_manifest.csv")
OUTPUT_ROOT = Path("../dataset_split")
RANDOM_SEED = 42

manifest = pd.read_csv(MANIFEST_PATH)
manifest["abs_path"] = manifest["path"].apply(lambda p: Path(p).resolve())
print(f"Total citra dalam manifest asli: {len(manifest)}")
manifest["label"].value_counts()


Total citra dalam manifest asli: 3680


label
cancer       1874
no_cancer    1806
Name: count, dtype: int64

## 1. Perceptual hashing — deteksi duplikat

In [2]:
hashes = []
for p in manifest["abs_path"]:
    with Image.open(p) as im:
        arr = np.array(im.convert("RGB").resize((64, 64)))
    hashes.append(hashlib.md5(arr.tobytes()).hexdigest())
manifest["phash"] = hashes

groups = manifest.groupby("phash")
print(f"Total grup citra unik (perceptual hash): {groups.ngroups} dari {len(manifest)} citra")
print(f"Grup dengan >1 salinan (duplikat): {(groups.size() > 1).sum()}")


Total grup citra unik (perceptual hash): 2886 dari 3680 citra
Grup dengan >1 salinan (duplikat): 742


## 2. Identifikasi & buang grup dengan label bertentangan

In [3]:
conflicting_hashes = set()
for h, g in groups:
    if g["label"].nunique() > 1:
        conflicting_hashes.add(h)

n_conflicting_images = manifest[manifest["phash"].isin(conflicting_hashes)].shape[0]
print(f"Grup dengan label bertentangan: {len(conflicting_hashes)} ({n_conflicting_images} citra dibuang)")

clean = manifest[~manifest["phash"].isin(conflicting_hashes)].copy()


Grup dengan label bertentangan: 11 (23 citra dibuang)


## 3. Deduplikasi — satu representatif per citra unik

In [4]:
deduped = clean.drop_duplicates(subset="phash", keep="first").reset_index(drop=True)
print(f"Setelah drop conflicting-label + dedup: {len(deduped)} citra unik")
print(deduped["label"].value_counts())
print("\nProporsi kelas (%):")
print((deduped["label"].value_counts(normalize=True) * 100).round(2))

deduped[["split", "label", "filename", "ext", "path", "phash"]].to_csv(
    "../reports/dataset_deduplicated_manifest.csv", index=False
)


Setelah drop conflicting-label + dedup: 2875 citra unik
label
no_cancer    1670
cancer       1205
Name: count, dtype: int64

Proporsi kelas (%):
label
no_cancer    58.09
cancer       41.91
Name: proportion, dtype: float64


## 4. Stratified split 70/15/15 pada data yang sudah bersih

In [5]:
train_df, temp_df = train_test_split(
    deduped,
    test_size=0.30,
    stratify=deduped["label"],
    random_state=RANDOM_SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_SEED,
)

train_df = train_df.copy()
train_df["new_split"] = "train"
val_df = val_df.copy()
val_df["new_split"] = "val"
test_df = test_df.copy()
test_df["new_split"] = "test"

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(split_df.groupby(["new_split", "label"]).size().unstack(fill_value=0))
print("\nProporsi split (%):")
print((split_df["new_split"].value_counts(normalize=True) * 100).round(2))


label      cancer  no_cancer
new_split                   
test          181        251
train         843       1169
val           181        250

Proporsi split (%):
new_split
train    69.98
test     15.03
val      14.99
Name: proportion, dtype: float64


## 5. Salin file ke folder `dataset_split/` (bersih dari duplikat)

In [6]:
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

for new_split in ["train", "val", "test"]:
    for label in ["cancer", "no_cancer"]:
        (OUTPUT_ROOT / new_split / label).mkdir(parents=True, exist_ok=True)

def dest_filename(row):
    return f"{row['split']}__{row['filename']}"

split_df["dest_filename"] = split_df.apply(dest_filename, axis=1)
split_df["dest_path"] = split_df.apply(
    lambda r: str(OUTPUT_ROOT / r["new_split"] / r["label"] / r["dest_filename"]), axis=1
)

for i, row in enumerate(split_df.itertuples(index=False), start=1):
    shutil.copy2(row.abs_path, row.dest_path)
    if i % 500 == 0:
        print(f"{i}/{len(split_df)} file disalin...")

print("Selesai menyalin semua file.")


500/2875 file disalin...


1000/2875 file disalin...


1500/2875 file disalin...


2000/2875 file disalin...


2500/2875 file disalin...


Selesai menyalin semua file.


## 6. Verifikasi hasil split

In [7]:
verify_counts = {}
for new_split in ["train", "val", "test"]:
    for label in ["cancer", "no_cancer"]:
        folder = OUTPUT_ROOT / new_split / label
        verify_counts[(new_split, label)] = len(list(folder.iterdir()))

verify_df = pd.Series(verify_counts).unstack()
print(verify_df)
print("\nTotal file di dataset_split/:", verify_df.values.sum())
print("Total citra unik setelah dedup:", len(deduped))
assert verify_df.values.sum() == len(deduped), "Jumlah file tidak cocok!"
print("Verifikasi OK — jumlah file cocok.")


       cancer  no_cancer
test      181        251
train     843       1169
val       181        250

Total file di dataset_split/: 2875
Total citra unik setelah dedup: 2875
Verifikasi OK — jumlah file cocok.


## 7. Simpan manifest split final

In [8]:
split_df[["new_split", "label", "filename", "dest_filename", "dest_path", "phash"]].to_csv(
    "../reports/dataset_split_manifest.csv", index=False
)
print("Manifest split disimpan ke reports/dataset_split_manifest.csv")
split_df[["new_split", "label"]].value_counts().sort_index()


Manifest split disimpan ke reports/dataset_split_manifest.csv


new_split  label    
test       cancer        181
           no_cancer     251
train      cancer        843
           no_cancer    1169
val        cancer        181
           no_cancer     250
Name: count, dtype: int64